<a href="https://colab.research.google.com/github/abdulsamad00529/ML-assignments-FlyRank-AI/blob/main/w02_ml_task_framing_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 1 — Ranking Signal Analysis.** Question: *which safe content and search signals are
associated with visibility, clicks, engagement, or movement?*

Mapped onto the four task types, this is a **classification** task: I predict a binary,
observed outcome — did this content item's traffic **decline** (`trend_direction == "down"`)
over the trailing 30 days vs. the prior 30 — from safe content/search signals. The predicted
label itself isn't the product I care about; the model is a tool to get honest **effect
sizes / feature importances** out of many tangled signals at once, which is what "signal
analysis" actually means in practice. A pure ranking or scoring framing doesn't fit as well
here because I'm not yet producing a prioritized action queue (that's Lane 2's job) — I'm
asking "which signals move with decline, and by how much," which is a classification-plus-
feature-importance question.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 50)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"{len(df):,} content items across {df['client_id'].nunique()} clients")
print()
print("trend_direction distribution (%):")
print((df["trend_direction"].value_counts(normalize=True) * 100).round(1))


30,000 content items across 32 clients

trend_direction distribution (%):
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64


## 2. Target or proxy

**Target:** `is_declining` = 1 if `trend_direction == "down"`, else 0.

This is an **observed** outcome, not a rule I invented: `trend_direction` is computed upstream
from `trend_pct`, which compares `impressions_last_30d` to `impressions_prev_30d` (>20% drop =
"down"). I'm reusing that measured label, not defining a new one.

**Guardrail (the label trap):** `trend_direction` and `trend_pct` — and the 30d/prev_30d
impression, click, and session columns they're built from — can **never** be features. They
either define the label directly or would leak the answer into the inputs.


In [ ]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining"].value_counts())
print(f"\nBase rate (share declining): {df['is_declining'].mean():.1%}")

df[["content_id", "trend_pct", "trend_direction", "is_declining"]].sample(5, random_state=7)


is_declining
1    16262
0    13738
Name: count, dtype: int64

Base rate (share declining): 54.2%


,content_id,trend_pct,trend_direction,is_declining
1252,content_a8c35eeef547,-6.0,stable,0
10444,content_26eb8cf5167c,-91.9,down,1
8994,content_663ee9569ab2,-18.2,stable,0
7463,content_7e13c4bf098b,NaN,flat,0
1910,content_b9b75b2367a3,-69.6,down,1


## 3. Success metric

**ROC-AUC**, measured against a majority-class baseline — not accuracy.

The base rate is ~54% declining, so a model that always predicts "declining" already scores
~54% accuracy without learning anything. ROC-AUC (threshold-independent, compares the model's
ranking of declining vs. non-declining items) is the metric I can defend as "good." I'll treat
0.50 as the floor (coin flip) and look for a real, non-trivial margin above it before I claim
any signal matters — plus a by-hand look at which features drive the ranking, since the
feature effects are the actual deliverable of this lane, not the predictions themselves.


In [ ]:
majority_share = df["is_declining"].mean()
majority_baseline_accuracy = max(majority_share, 1 - majority_share)

print(f"Majority-class baseline accuracy: {majority_baseline_accuracy:.1%}  <- misleading, not my metric")
print("ROC-AUC floor (coin flip): 0.50  <- what I need to beat, by a real margin")


Majority-class baseline accuracy: 54.2%  <- misleading, not my metric
ROC-AUC floor (coin flip): 0.50  <- what I need to beat, by a real margin


## 4. The unit of analysis, as a real dataframe

**One row = one content item** (`content_id`) with its trailing-90-day content and search
signals, pseudonymized to a `client_id`. That's the starter dataset's native grain, and it's
exactly the grain this lane needs: I'm comparing content items to each other, not clients or
days.


In [ ]:
safe_features = [
    "content_type", "main_intent",
    "word_count", "char_count", "word_count_tier", "char_count_tier",
    "search_volume", "competition", "competition_level", "cpc",
    "content_age_days", "age_tier", "days_since_last_update", "freshness_tier",
    "avg_position", "position_tier", "impression_tier",
]
target = "is_declining"

lane_df = df[["content_id", "client_id"] + safe_features + [target]].copy()

print(f"Unit of analysis: one row = one content item")
print(f"{lane_df.shape[0]:,} rows x {lane_df.shape[1]} columns")
lane_df.head()


Unit of analysis: one row = one content item
30,000 rows x 20 columns


,content_id,client_id,content_type,main_intent,word_count,char_count,word_count_tier,char_count_tier,search_volume,competition,competition_level,cpc,content_age_days,age_tier,days_since_last_update,freshness_tier,avg_position,position_tier,impression_tier,is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,20457.0,2000-3500,15000-25000,10.0,0.67,HIGH,2.05,187,181-365,20,0-30,10.6,striking,good,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,15562.0,2000-3500,15000-25000,90.0,0.01,LOW,0.05,445,365+,25,0-30,20.3,page_3_5,good,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,23643.0,3500+,15000-25000,0.0,0.00,LOW,0.00,141,91-180,20,0-30,36.5,page_3_5,good,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,NaN,NaN,NaN,10.0,0.00,LOW,0.00,463,365+,22,0-30,6.2,page_1,good,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,17469.0,2000-3500,15000-25000,0.0,0.00,LOW,0.00,263,181-365,14,0-30,44.0,page_3_5,good,1


## 5. Why ML beats a fixed rule here

A fixed rule needs one clean threshold on one signal. None exists here — every single signal
I checked has a wide, overlapping spread of decline rates, not a clean split:

- `position_tier`: decline rate ranges from **24.1%** (`top_3`) to **61.0%** (`striking`) —
  the biggest single-signal spread I found, and still nowhere near a clean if/else.
- `freshness_tier`: **47.1%–61.1%** across tiers — barely moves at all on its own.
- `content_type`: **28.7%–57.2%** — `feedly article` looks safer, but the other two types are
  close to a coin flip.

No single column separates decliners from non-decliners; the tiers overlap heavily and the
signals likely interact (e.g., a `striking`-position page might behave differently at
`0-30` freshness vs `91-180`). That's exactly the "many signals, tangled, shifting" case the
framing skill flags as where ML earns its place — a model can weigh and combine all of these
at once and report which ones matter most, honestly, instead of me guessing one threshold and
hard-coding it.


In [ ]:
for col in ["position_tier", "freshness_tier", "content_type"]:
    print(f"decline rate by {col}:")
    print(lane_df.groupby(col)[target].agg(["mean", "count"]).round(3))
    print()


decline rate by position_tier:
                mean  count
position_tier              
deep           0.344   1319
page_1         0.570  11814
page_3_5       0.562   7242
striking       0.610   7304
top_3          0.241   2321

decline rate by freshness_tier:
                 mean  count
freshness_tier              
0-30            0.511  20480
181+            0.471    174
31-90           0.589    175
91-180          0.611   9171

decline rate by content_type:
                     mean  count
content_type                    
comparison article  0.572    697
feedly article      0.287   2096
keyword article     0.561  27207



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.